# 01. Caracterización de datos

Consulta el perfilamiento de Bronze generado por `src/caracterizar_bookeau.py` y `src/documentar_caracterizacion.py` (pasos `caracterizar` y `documentar` de `main.py`). Resultados en `docs/artifacts/caracterizacion/`.

Con los datos reales: 114 libros de reservas y encuestas, 86.562 reservas con ID único y 60.390 encuestas vinculadas (146.952 filas en total, que no equivalen a reservas).

**Antes de ejecutar:** desde la raíz del proyecto, correr `python3 src/main.py --dummy` (datos ficticios versionados) o `python3 src/main.py` (datos reales, que no se versionan). El cuaderno solo lee los resultados del pipeline en `docs/artifacts/`, `data/plata/` y `data/oro/`. Con datos ficticios los conteos son pequeños y no coinciden con las cifras del informe, que usan los datos reales.


In [1]:
from pathlib import Path
import csv
from html import escape
from IPython.display import display, HTML

PROYECTO = next(
    (p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "main.py").is_file()),
    None,
)
if PROYECTO is None:
    raise RuntimeError("Abre el cuaderno desde la carpeta del proyecto o desde notebooks/.")
RESULTADOS = PROYECTO / "docs" / "artifacts" / "caracterizacion"
if not RESULTADOS.is_dir():
    raise RuntimeError("Faltan los resultados del pipeline. Ejecuta primero desde la raíz: "
                       "python3 src/main.py --dummy  (o sin --dummy si tienes los datos reales)")

def cargar(nombre):
    with (RESULTADOS / nombre).open(newline="", encoding="utf-8") as archivo:
        return list(csv.DictReader(archivo))

def tabla(filas, columnas=None, limite=None):
    if not filas:
        print("Sin registros para esta selección.")
        return
    columnas = columnas or list(filas[0])
    mostrar = filas if limite is None else filas[:limite]
    cabecera = "".join(f"<th>{escape(c)}</th>" for c in columnas)
    cuerpo = "".join(
        "<tr>" + "".join(f"<td>{escape(str(fila[c]))}</td>" for c in columnas) + "</tr>"
        for fila in mostrar
    )
    display(HTML(f"<table><thead><tr>{cabecera}</tr></thead><tbody>{cuerpo}</tbody></table>"))
    print(f"Mostrando {len(mostrar)} de {len(filas)} registros.")


## 1. Volumen, grano y cobertura

Una fila de Reservas corresponde a un evento de reserva. Una fila de encuesta corresponde a una respuesta asociada al evento, según el ID observado. No se deben sumar las encuestas al conteo de reservas.

| Fuente | Filas | Variables |
|---|---:|---:|
| Reservas | 86.562 | 17 |
| Encuesta Reserva Express | 13.815 | 22 |
| Encuesta Reserva Normal IP | 23.437 | 22 |
| Encuesta Express | 6.759 | 33 |
| Encuesta Normal | 16.379 | 35 |


In [2]:
tabla(cargar("resumen_grupos.csv"))
archivos = cargar("archivos.csv")
tabla([f for f in archivos if int(f["filas"]) == 0])


grupo,archivos,filas,variables,variantes_encabezado,id_distintos,id_faltantes,filas_excedentes_id_repetido,filas_excedentes_id_periodo_repetido,filas_excedentes_duplicado_exacto,inicio_min,inicio_max,duracion_programada_min_minutos,duracion_programada_mediana_minutos,duracion_programada_max_minutos
Encuesta Express,9,168,33,1,168,0,0,0,0,2017-02-08 09:00:00.000000,2021-11-15 09:00:00.000000,30.0,30.0,50.0
Encuesta Normal,9,437,35,1,437,0,0,0,0,2017-02-01 13:30:00.000000,2021-11-10 08:00:00.000000,30.0,60.0,120.0
Encuesta Reserva Express,4,107,22,1,107,0,0,0,0,2019-08-01 13:30:00.000000,2021-11-16 15:30:00.000000,30.0,30.0,30.0
Encuesta Reserva Normal IP,4,215,22,1,215,0,0,0,0,2019-08-01 17:30:00.000000,2021-11-19 17:00:00.000000,60.0,60.0,60.0
Reservas,9,1972,17,1,1972,0,0,0,0,2017-02-01 12:00:00.000000,2021-11-19 17:00:00.000000,30.0,60.0,120.0


Mostrando 5 de 5 registros.
Sin registros para esta selección.


## 2. Variables y significado inicial

Las descripciones son interpretaciones de los encabezados y del README. Confirmar reglas operativas, escalas y catálogos con el responsable de la fuente. Consultar `diccionario_variables.md` para las preguntas completas.


In [3]:
variables = cargar("variables.csv")
diccionario = cargar("diccionario_variables.csv")
tabla([v for v in diccionario if v["grupo"] == "Reservas"],
      ["variable", "descripcion_inicial", "tipos_observados", "porcentaje_faltantes", "valores_distintos"])
# Cambiar el grupo para explorar las preguntas y tipos de cada encuesta.
GRUPO = "Encuesta Express"
tabla([v for v in diccionario if v["grupo"] == GRUPO],
      ["variable", "rol", "tipos_observados", "porcentaje_faltantes", "primer_periodo_con_valores"])


variable,descripcion_inicial,tipos_observados,porcentaje_faltantes,valores_distintos
ID,Identificador del evento de reserva. Único en cada grupo; todas las encuestas lo comparten con Reservas.,"{""int"": 1972}",0.0,1972
Fecha inicio,"Inicio programado del evento, interpretado a partir del encabezado y del formato Excel.","{""datetime"": 1972}",0.0,1781
Fecha fin,Fin programado del evento. No equivale a una salida real registrada.,"{""datetime"": 1972}",0.0,1805
Fecha de llegada,Marca temporal de llegada según el encabezado; confirmar procedimiento de captura y excepciones.,"{""datetime"": 989}",49.8479,986
Fecha de devolución,Campo temporal sin valores en todos los archivos; su significado operativo debe confirmarse.,{},100.0,0
Estado de la reserva,"Estado del evento de reserva, con 12 etiquetas en la fuente principal. No confundir con Estado de la encuesta.","{""str"": 1972}",0.0,12
Prioritaria,Indicador sí/no de prioridad de la reserva; confirmar regla de asignación.,"{""str"": 1972}",0.0,2
Categoría,"Categoría de recurso o prestación, como Tutor Presencial; incluye identificadores históricos Deprecated.","{""str"": 1972}",0.0,5
Tipo de horario,"Modalidad o tipo de agenda: Normal, Express, Normal pico y otras modalidades históricas.","{""str"": 1972}",0.0,7
Servicio,"Servicio o curso asociado: IP, EDA y etiquetas históricas, entre otros.","{""str"": 1972}",0.0,7


Mostrando 17 de 17 registros.


variable,rol,tipos_observados,porcentaje_faltantes,primer_periodo_con_valores
ID,identificador,"{""int"": 168}",0.0,201710
Fecha inicio,fecha/hora,"{""datetime"": 168}",0.0,201710
Fecha fin,fecha/hora,"{""datetime"": 168}",0.0,201710
Fecha de llegada,fecha/hora,"{""datetime"": 166}",1.1905,201710
Fecha de devolución,fecha/hora,{},100.0,
Estado de la reserva,categoría o respuesta,"{""str"": 168}",0.0,201710
Prioritaria,categoría o respuesta,"{""str"": 168}",0.0,201710
Categoría,categoría o respuesta,"{""str"": 168}",0.0,201710
Tipo de horario,categoría o respuesta,"{""str"": 168}",0.0,201710
Servicio,categoría o respuesta,"{""str"": 168}",0.0,201710


Mostrando 33 de 33 registros.


## 3. Completitud y coherencia temporal

`Fecha de devolución` está vacía en todas las fuentes: no hay base para calcular duración efectiva. `Fecha de llegada` falta en el 50,16% de Reservas; se debe interpretar junto con el estado. Algunas preguntas no tienen respuestas en los primeros períodos, aunque sus encabezados estén presentes.


In [4]:
tabla([v for v in variables if v["grupo"] == "Reservas"],
      ["variable", "filas", "no_vacios", "faltantes", "porcentaje_faltantes"])
tabla(cargar("estado_llegada.csv"))
tabla(cargar("reglas_fechas.csv"))
tabla(cargar("reglas_respuestas.csv"))


variable,filas,no_vacios,faltantes,porcentaje_faltantes
ID,1972,1972,0,0.0
Fecha inicio,1972,1972,0,0.0
Fecha fin,1972,1972,0,0.0
Fecha de llegada,1972,989,983,49.8479
Fecha de devolución,1972,0,1972,100.0
Estado de la reserva,1972,1972,0,0.0
Prioritaria,1972,1972,0,0.0
Categoría,1972,1972,0,0.0
Tipo de horario,1972,1972,0,0.0
Servicio,1972,1972,0,0.0


Mostrando 17 de 17 registros.


estado_reserva,filas,con_llegada,sin_llegada,porcentaje_con_llegada
Cancelada,23,0,23,0.0
Cancelada a tiempo,416,4,412,0.9615
Cancelada con excusa,32,0,32,0.0
Cancelada por sanción,49,0,49,0.0
Cancelada y sancionada,12,0,12,0.0
Cola cancelada por reservación,102,0,102,0.0
En cola,230,0,230,0.0
En ejecución,2,2,0,100.0
Finalizada,888,880,8,99.0991
No asistió,115,1,114,0.8696


Mostrando 12 de 12 registros.


grupo,regla,evaluables,incumplimientos,porcentaje_evaluables
Encuesta Express,fin_anterior_a_inicio,168,0,0.0
Encuesta Express,devolucion_anterior_a_llegada,0,0,
Encuesta Express,inicio_fuera_anio_periodo,168,0,0.0
Encuesta Normal,fin_anterior_a_inicio,437,0,0.0
Encuesta Normal,devolucion_anterior_a_llegada,0,0,
Encuesta Normal,inicio_fuera_anio_periodo,437,0,0.0
Encuesta Reserva Express,fin_anterior_a_inicio,107,0,0.0
Encuesta Reserva Express,devolucion_anterior_a_llegada,0,0,
Encuesta Reserva Express,inicio_fuera_anio_periodo,107,0,0.0
Encuesta Reserva Normal IP,fin_anterior_a_inicio,215,0,0.0


Mostrando 15 de 15 registros.


grupo,variable,regla,no_vacios_evaluados,fuera_dominio,faltantes
Encuesta Express,Califique la ayuda que le dio su tutor,valor entero entre 1 y 5 (dominio candidato observado),108,0,60
Encuesta Normal,Califique la ayuda que le dio su tutor,valor entero entre 1 y 5 (dominio candidato observado),268,0,169


Mostrando 2 de 2 registros.


## 4. Identificadores y vínculos

Todos los ID de encuesta existen en Reservas y tienen el mismo período. No hay ID repetidos dentro de cada grupo. Comparar los atributos repetidos antes de decidir la fuente de referencia.


In [5]:
tabla(cargar("vinculos_reservas_encuestas.csv"))
tabla([r for r in cargar("consistencia_entre_fuentes.csv") if int(r["diferencias"]) > 0])


grupo_encuesta,filas_con_id,id_distintos,id_en_reservas,filas_con_id_en_reservas,filas_sin_id_en_reservas,filas_con_id_y_periodo_en_reservas,porcentaje_filas_con_id_en_reservas
Encuesta Express,168,168,168,168,0,168,100.0
Encuesta Normal,437,437,437,437,0,437,100.0
Encuesta Reserva Express,107,107,107,107,0,107,100.0
Encuesta Reserva Normal IP,215,215,215,215,0,215,100.0


Mostrando 4 de 4 registros.


grupo_encuesta,variable,pares_no_vacios_comparados,diferencias,porcentaje_diferencias
Encuesta Express,Prestador del servicio,168,1,0.5952
Encuesta Normal,Programa,425,1,0.2353
Encuesta Normal,Prestador del servicio,437,3,0.6865
Encuesta Reserva Normal IP,Programa,207,2,0.9662


Mostrando 4 de 4 registros.


## 5. Categorías y dominios

`Estado` es la validez de una encuesta; `Estado de la reserva` describe el evento. No confundirlos al definir indicadores. Las variantes de tildes y mayúsculas son candidatos de homologación; los catálogos Deprecated requieren explicación histórica.


In [6]:
categorias = cargar("categorias.csv")
tabla([r for r in categorias if r["grupo"] == "Reservas" and r["variable"] in
       {"Estado de la reserva", "Tipo de horario", "Servicio"}])
tabla([r for r in categorias if r["variable"] == "Estado"])
tabla([r for r in cargar("variantes_categoricas.csv") if r["grupo"] == "Reservas"])


grupo,variable,valor,frecuencia,porcentaje_filas
Reservas,Estado de la reserva,Finalizada,888,45.0304
Reservas,Estado de la reserva,Cancelada a tiempo,416,21.0953
Reservas,Estado de la reserva,En cola,230,11.6633
Reservas,Estado de la reserva,No asistió,115,5.8316
Reservas,Estado de la reserva,Cola cancelada por reservación,102,5.1724
Reservas,Estado de la reserva,Realizada,96,4.8682
Reservas,Estado de la reserva,Cancelada por sanción,49,2.4848
Reservas,Estado de la reserva,Cancelada con excusa,32,1.6227
Reservas,Estado de la reserva,Cancelada,23,1.1663
Reservas,Estado de la reserva,Cancelada y sancionada,12,0.6085


Mostrando 26 de 26 registros.


grupo,variable,valor,frecuencia,porcentaje_filas
Encuesta Express,Estado,Válida,165,98.2143
Encuesta Express,Estado,Inválida,3,1.7857
Encuesta Normal,Estado,Válida,432,98.8558
Encuesta Normal,Estado,Inválida,5,1.1442
Encuesta Reserva Express,Estado,Válida,106,99.0654
Encuesta Reserva Express,Estado,Inválida,1,0.9346
Encuesta Reserva Normal IP,Estado,Válida,215,100.0


Mostrando 7 de 7 registros.


grupo,variable,valor_normalizado_candidato,variantes_originales_json,filas
Reservas,Programa,ingenieria electronica,"[[""Ingeniería Electrónica"", 198], [""INGENIERIA ELECTRONICA"", 211]]",409
Reservas,Programa,ingenieria de sistemas,"[[""Ingeniería de Sistemas"", 196], [""INGENIERÍA DE SISTEMAS"", 173], [""INGENIERIA DE SISTEMAS"", 182]]",551


Mostrando 2 de 2 registros.


## 6. Texto libre

Las encuestas contienen comentarios sobre tutor y tutoría, observaciones del curso, sugerencias y descripciones del uso de IA. Se contabilizan respuestas no vacías y longitudes; no se asume que toda respuesta sea sustantiva.


In [7]:
tabla(cargar("texto_libre.csv"))


grupo,variable,respuestas_no_vacias,porcentaje_cobertura,textos_distintos,longitud_min,longitud_mediana,longitud_max
Encuesta Express,Indique cuál fue la herramienta usada durante la tutoría (si sabe cuál es) y el uso que tuvo la misma en la tutoría.,0,0.0,0,,,
Encuesta Express,Comente aspectos positivos y negativos del tutor y la tutoría. Recuerde que las respuestas van a ser anónimas.,131,77.9762,14,3,27,49
Encuesta Express,Escriba en este recuadro cualquier observación o comentario relacionado al curso y sus contenidos.,15,8.9286,10,3,22,49
Encuesta Express,"Sugerencias, reclamos y observaciones",46,27.381,6,7,25.0,34
Encuesta Normal,Indique cuál fue la herramienta usada durante la tutoría (si sabe cuál es) y el uso que tuvo la misma en la tutoría.,1,0.2288,1,45,45,45
Encuesta Normal,Comente aspectos positivos y negativos del tutor y la tutoría. Recuerde que las respuestas van a ser anónimas.,333,76.2014,14,3,22,49
Encuesta Normal,Escriba en este recuadro cualquier observación o comentario relacionado al curso y sus contenidos.,45,10.2975,14,3,22,49
Encuesta Normal,"Sugerencias, reclamos y observaciones",116,26.5446,6,7,26.0,34


Mostrando 8 de 8 registros.


## 7. Cobertura por período

La primera respuesta observada no demuestra cuándo se creó la pregunta. Antes de comparar satisfacción históricamente, revisar formularios y denominadores de respuesta.


In [8]:
cobertura = cargar("cobertura_variables_periodo.csv")
PREGUNTA = "Califique la ayuda que le dio su tutor"
tabla([r for r in cobertura if r["grupo"] == GRUPO and r["variable"] == PREGUNTA])


grupo,periodo,variable,filas,no_vacios,porcentaje_no_vacios
Encuesta Express,201710,Califique la ayuda que le dio su tutor,20,0,0.0
Encuesta Express,201719,Califique la ayuda que le dio su tutor,12,0,0.0
Encuesta Express,201720,Califique la ayuda que le dio su tutor,21,0,0.0
Encuesta Express,201819,Califique la ayuda que le dio su tutor,25,23,92.0
Encuesta Express,201820,Califique la ayuda que le dio su tutor,24,22,91.6667
Encuesta Express,201920,Califique la ayuda que le dio su tutor,17,15,88.2353
Encuesta Express,202010,Califique la ayuda que le dio su tutor,18,18,100.0
Encuesta Express,202110,Califique la ayuda que le dio su tutor,17,17,100.0
Encuesta Express,202120,Califique la ayuda que le dio su tutor,14,13,92.8571


Mostrando 9 de 9 registros.


## 8. Regenerar la caracterización

Las celdas anteriores leen los resultados existentes. La siguiente celda vuelve a leer todos los Excel y genera las tablas y documentos; activarla cuando se incorporen fuentes nuevas. Usa la biblioteca estándar de Python y conserva los originales.


In [9]:
# import runpy
# runpy.run_path(str(PROYECTO / "src" / "caracterizar_bookeau.py"), run_name="__main__")
# runpy.run_path(str(PROYECTO / "src" / "documentar_caracterizacion.py"), run_name="__main__")


## Decisiones pendientes

Confirmar estados de reserva, reglas de asistencia, códigos de período, categorías históricas, identidad de participantes, escalas y formularios históricos, y criterios de invalidez de encuestas.

La fuente permite explorar demanda y satisfacción. No incluye notas académicas, capacidad disponible explícita ni costos para demostrar impacto causal de tutorías, ocupación de capacidad o rentabilidad.
